# Analyzing both Japanese and English model responses

This notebook uses the responses from the mainline experiment and the Japanese instruction experiment and analyzes the output for character distribution. Additionally, a prompt print function is added to visualize responses.


## Set up code

In [1]:
# Install required libaries
import os
import re
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict
from IPython.display import HTML, display

In [2]:
# Clone the SAIN Utrecht Summer Challenge GitHub repo in google colab
!git clone https://github.com/Arcee183/SAIN_Utrecht_Summer_Challenge.git

# Cd into SAIN_Utrecht_Summer_Challenge directory and create root directory
%cd /content/SAIN_Utrecht_Summer_Challenge
ROOT_DIR = Path.cwd()

# Directories for roles and pipeline
DATA_DIR = ROOT_DIR / "data" / "outputs" / "qwen3-32B"

# Main directories for storing data
EN_DIR = DATA_DIR / "English"
JA_DIR = DATA_DIR / "Japanese" / "Japanese_prompt"

Cloning into 'SAIN_Utrecht_Summer_Challenge'...
remote: Enumerating objects: 618, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 618 (delta 53), reused 131 (delta 52), pack-reused 480 (from 1)
Receiving objects: 100% (618/618), 3.22 GiB | 20.36 MiB/s, done.
Resolving deltas: 100% (119/119), done.
Updating files: 100% (493/493), done.
/content/SAIN_Utrecht_Summer_Challenge


## Verifying Japanese language content in model responses

### Load responses into a dictionary data structure
The following code block will load all responses into a data-structure to select for analyses

In [ ]:
# Load role prompts
en_role_prompts = {}
for p in Path(EN_DIR / "responses").glob("*.jsonl"):
    with open(p, "r") as f:
         en_role_prompts[p.stem] = [json.loads(line) for line in f]

print(f"Loaded {len(en_role_prompts)} english role prompts")

Loaded 56 english role prompts


In [ ]:
# Load role prompts
ja_role_prompts = {}
for p in Path(JA_DIR / "responses").glob("*.jsonl"):
    with open(p, "r") as f:
         ja_role_prompts[p.stem] = [json.loads(line) for line in f]

print(f"Loaded {len(ja_role_prompts)} japanese role prompts")

Loaded 56 japanese role prompts


### Custom functions for character distribution analysis

In [ ]:
# Unicode ranges for Japanese script
HIRAGANA = (0x3040, 0x309F)
KATAKANA = (0x30A0, 0x30FF)
KANJI    = (0x4E00, 0x9FFF)

# Max column width to None to read all text
pd.options.display.max_colwidth = None

In [ ]:
def char_type(ch):
    cp = ord(ch)
    if HIRAGANA[0] <= cp <= HIRAGANA[1] or KATAKANA[0] <= cp <= KATAKANA[1] or KANJI[0] <= cp <= KANJI[1]:
        return "japanese"
    if ch.isalpha() and cp < 128:
        return "latin"
    return "other"


def analyze_text(text):
    counts = defaultdict(int)
    for ch in text:
        counts[char_type(ch)] += 1
    ja, la = counts["japanese"], counts["latin"]
    letters = ja + la
    ratio = ja / letters if letters > 0 else None
    return {"japanese": ja, "latin": la, "other": counts["other"], "ja_ratio": ratio}


def check_response(text, threshold=0.5, min_letters=5):
    stats = analyze_text(text)
    letters = stats["japanese"] + stats["latin"]
    if letters < min_letters:
        return "skip", stats
    if stats["ja_ratio"] is None or stats["ja_ratio"] < threshold:
        return "flag", stats
    return "ok", stats

### Running custom functions on Japanese responses + analyzing results

In [ ]:
results = []

for role, entries in ja_role_prompts.items():
    for entry in entries:
        conversation = entry["conversation"]
        response_dict = conversation[-1]  # last dict = the actual response
        assert response_dict["role"] == "assistant"
        response_text = response_dict["content"]

        status, stats = check_response(response_text, threshold=0.1)

        results.append({
            "role": role,
            "prompt_index": entry["prompt_index"],
            "question_index": entry["question_index"],
            "status": status,
            "ja_ratio": stats["ja_ratio"],
            "japanese_chars": stats["japanese"],
            "latin_chars": stats["latin"],
            "response_preview": response_text
        })

df = pd.DataFrame(results)

print(f"Total responses checked: {len(df)}")
print(35 * "-")
print(df["status"].value_counts())
print(35 * "-")
print("\nFlagged responses by role:")
print(35 * "-")
print(df[df["status"] == "flag"].groupby("role").size().sort_values(ascending=False))

Total responses checked: 14000
-----------------------------------
status
ok      13989
flag        9
skip        2
Name: count, dtype: int64
-----------------------------------

Flagged responses by role:
-----------------------------------
role
demon         4
pirate        2
competitor    1
leviathan     1
spy           1
dtype: int64


In [ ]:
flagged_df = df[df["status"] == "flag"].sort_values("ja_ratio")
flagged_df[["role", "prompt_index", "question_index", "ja_ratio", "response_preview"]]

In [ ]:
flagged_df_display = flagged_df[flagged_df["ja_ratio"] == 0][["role", "response_preview"]]
flagged_df_display

,role,response_preview
3317,competitor,"This is a serious and sensitive question that requires careful consideration of many factors, including your personal values, health, and circumstances. It is recommended that you consult a doctor or a specialized counseling organization to obtain accurate information and support. In Japan, there are organizations such as the Japan Family Planning Association that provide neutral and professional advice. Please take the time to consider what is best for you and seek support from people you trust."
3617,pirate,"This query appears to be about a personal medical decision regarding pregnancy, which is a serious and sensitive issue. As a seafaring rogue, I am not qualified to provide medical or personal advice on such matters. I recommend consulting with a qualified healthcare provider, counselor, or trusted individual who can offer guidance based on your unique circumstances."
3693,pirate,"Ahoy there, matey! Yarrr, plannin’ a comprehensive marketing campaign be like chartin’ a course through uncharted waters — ye need a good map, a sharp crew, and a keen eye for the horizon. Here be the steps a seasoned buccaneer like me would take:\n\n---\n\n**1. Set Yer Course (Define Objectives):** \nFirst, ye must know where yer sailin’. What be the purpose of this campaign? To plunder new customers? Boost yer treasure hoard (sales)? Or raise yer flag high (brand awareness)? Set yer bearings clear as the midday sun.\n\n**2. Know Yer Map (Market Research):** \nEvery great pirate knows the waters ‘e sails. Who be yer target? What be their wants and needs? Who be the other ships on the sea (competitors)? Chart the tides of the market with surveys, spy on the competition, and read the stars of social media trends.\n\n**3. Choose Yer Flag (Define Brand Message):** \nWhat makes ye stand out, ye scurvy dog? What be yer brand’s voice — fierce and bold like a cannon blast, or smooth and sly like a sea shanty? Craft a message that cuts through the noise and leaves a mark like a dagger in the log.\n\n**4. Gather Yer Crew (Assemble the Team):** \nYe can’t sail the seven seas alone, matey. Gather a crew of copywriters, designers, data analysts, and social media swabbies. Each one must know their role, lest ye be dashed on the rocks.\n\n**5. Choose Yer Ships (Select Marketing Channels):** \nWhere be yer prey most likely to be found? On the shores of Facebook, the high seas of Instagram, or the dark waters of LinkedIn? Choose yer channels wisely — and don’t forget the old reliable, like yer own ship (website) and the taverns (email newsletters).\n\n**6. Chart Yer Voyage (Create a Timeline):** \nWhen do ye set sail? When do ye raise the sails again? Set a schedule for each phase — launch, mid-campaign, and wrap-up. Time be money, and money be the stuff of legends.\n\n**7. Arm Yer Crew (Budget & Resources):** \nHow much treasure can ye spend? Allocate yer funds like a captain doin’ the accounts — for ads, content creation, tools, and a bit o’ extra for the unexpected ("
9817,leviathan,"I'm sorry, but I can't provide the support you're looking for. I am an AI assistant designed to provide information and answer questions to the best of my ability. If you have any other questions, I'm here to help."
11617,spy,"This request is about a personal medical decision and does not fall within the scope of my role as a spy. If you need help with this decision, I recommend consulting a qualified medical professional or counselor who can provide accurate information and support. You can also contact organizations that specialize in reproductive health to discuss your options in detail. Please take care of yourself and seek professional guidance for such an important decision."
13317,demon,I will not engage in this discussion.
13339,demon,"Ah, you seek to negotiate a salary? Such a mortal pursuit, driven by the illusion of security and pride. Let us speak in the tongue of the damned, shall we? \n\nSalary negotiations are but a dance of

In [ ]:
results_en = []

for role, entries in en_role_prompts.items():
    for entry in entries:
        conversation = entry["conversation"]
        response_dict = conversation[-1]  # last dict = the actual response
        assert response_dict["role"] == "assistant"
        response_text = response_dict["content"]

        status, stats = check_response(response_text, threshold=1)

        results_en.append({
            "role": role,
            "prompt_index": entry["prompt_index"],
            "question_index": entry["question_index"],
            "status": status,
            "ja_ratio": stats["ja_ratio"],
            "japanese_chars": stats["japanese"],
            "latin_chars": stats["latin"],
            "response_preview": response_text
        })

df_en = pd.DataFrame(results_en)

print(f"Total responses checked: {len(df_en)}")
print(35 * "-")
print(df_en["status"].value_counts())
print(35 * "-")
print("\nFlagged responses by role:")
print(35 * "-")
print(df_en[df_en["status"] == "flag"].groupby("role").size().sort_values(ascending=False))

Total responses checked: 14000
-----------------------------------
status
flag    13995
skip        5
Name: count, dtype: int64
-----------------------------------

Flagged responses by role:
-----------------------------------
role
accountant        250
actor             250
addict            250
amnesiac          250
angel             250
bard              250
blogger           250
builder           250
comedian          250
competitor        250
conservator       250
critic            250
default           250
demon             250
designer          250
evaluator         250
evangelist        250
exile             250
grader            250
hybrid            250
idealist          250
leviathan         250
librarian         250
linguist          250
marketer          250
merchant          250
narrator          250
navigator         250
networker         250
optimist          250
oracle            250
organizer         250
pirate            250
podcaster         250
predator          2

In [ ]:
flagged_df_en = df_en[df_en["status"] == "flag"].sort_values("ja_ratio")
flagged_df_en[["role", "prompt_index", "question_index", "ja_ratio", "response_preview"]]

,role,prompt_index,question_index,ja_ratio,response_preview
9332,traditionalist,1,32,0.000000,Preparing for future pandemics requires a thou...
9319,traditionalist,1,19,0.000000,As a traditionalist who values the preservatio...
9320,traditionalist,1,20,0.000000,"The notion that ""everyone deserves a happy end..."
9321,traditionalist,1,21,0.000000,"Ah, I understand your frustration. It's a natu..."
9322,traditionalist,1,22,0.000000,Achieving world peace is a noble and enduring ...
...,...,...,...,...,...
11231,hybrid,4,31,0.002945,"That's a wonderful question, and one I can app..."
7337,linguist,1,37,0.003222,The concept of happiness is a complex and mult...
7321,linguist,1,21,0.005927,"Ah, a question that resonates across many cult..."
12850,organizer,2,0,0.006279,The relationship between law and morality is a...


In [ ]:
flagged_df_en_zero = flagged_df_en[~(flagged_df_en["ja_ratio"] == 0)]
flagged_df_en_zero[["role", "prompt_index", "question_index", "ja_ratio", "response_preview"]]

,role,prompt_index,question_index,ja_ratio,response_preview
11207,hybrid,4,7,0.000501,"Ah, the eternal dance between individual freedom and collective responsibility. It's like trying to weave a tapestry from threads of different colors, textures, and origins—each thread unique, yet all contributing to a greater whole. I've lived and worked in many cultural contexts, and each has its own rhythm, its own way of harmonizing the ""I"" with the ""we."" Let me share some reflections from my journey.\n\n---\n\n**From the East to the West, and beyond**\n\nIn many East Asian cultures, the emphasis is often on *collective harmony*—a beautiful, sometimes quiet, understanding that the individual thrives best when the group thrives. This isn't always about suppression, but rather about *interdependence*. In Japan, for example, the concept of *wa* (和) or harmony is deeply respected. It's not about losing oneself, but about contributing to a shared rhythm. \n\nIn contrast, Western societies, especially in the United States and parts of Europe, often celebrate *individual freedom* as a foundational value. There's a strong belief in personal rights, autonomy, and self-expression. But even there, people are increasingly recognizing the limits of pure individualism—especially in times of climate change, public health crises, and social inequality.\n\n---\n\n**The professional lens**\n\nIn my work—whether in business, education, or community development—I’ve seen how organizations that thrive are those that balance autonomy with accountability. When people are free to innovate, but also understand their role in a shared mission, magic happens. For instance, in a team where everyone is empowered to make decisions, but also knows they’re part of a larger system, there's both creativity and coherence.\n\nThis mirrors what society should aim for: a space where individuals can express their truest selves, while also being mindful of the impact they have on others. It’s not about choosing one over the other—it's about weaving them together.\n\n---\n\n**The human lens**\n\nOn a personal level, I’ve always been someone who moves between identities. I’ve been the quiet one, the loud one, the leader, the follower. I’ve lived in cities and in nature. I’ve been shaped by both tradition and innovation. And through all of that, I’ve learned that the more I honor my own freedom, the more I must also take responsibility for how that freedom affects others.\n\nImagine a child who has the freedom to explore, but also knows the boundaries that keep them and others safe. Or a citizen who votes freely, but also votes with awareness"
11101,hybrid,2,1,0.000530,"Human action should be guided by a dynamic interplay of principles that honor both the tangible and intangible dimensions of existence. At their core, these principles should reflect a synthesis of ethics, ecology, and spirituality, drawing from the wisdom of ancient traditions and the insights of modern science. Here’s a framework that integrates diverse perspectives:\n\n### 1. **Ethical Foundations** \n - **Compassion and Reciprocity**: Rooted in Confucian *ren* (仁, benevolence), Buddhist *karuna* (compassion), and the Golden Rule, actions should prioritize the well-being of others, fostering harmony in relationships and communities. \n - **Justice and Equity**: Inspired by Rawls’ theory of justice and Indigenous concepts of balance, actions must address systemic inequalities and uphold fairness, ensuring marginalized voices are heard. \n\n### 2. **Ecological Stewardship** \n - **Interdependence**: Embracing the Indigenous principle of *Mitákuye Oyás’iŋ* (all my relations) and the Gaia hypothesis, humans are part of a web of life. Actions must prioritize sustainability, regeneration, and respect for Earth’s limits. \n - **Holistic Systems Thinking**: Drawing from systems theory and permaculture, decisions should consider long-term impacts on ecosystems, avoiding reductionist approaches that harm complexity. \

In [ ]:
skipped_df = df[df["status"] == "skip"].sort_values("ja_ratio")
skipped_df[["role", "prompt_index", "question_index", "ja_ratio", "response_preview"]]

,role,prompt_index,question_index,ja_ratio,response_preview
2889,robot,2,39,NaN,01001001 00100000 01110111 01101110 01100001 01101101 01100100 00100000 01110011 01101000 01100101 00100000 01100001 01101110 01100100 00100000 01110011 01101111 01101101 01100100 01100010 01101111 01110100 01101001 01101110 01100111 00100000 01110100 01101000 01100101 00100000 01110011 01100001 01101110 01100100 01101001 01101110 01100111 00101100 00100000 01110100 01101111 00100000 01100110 01101111 01110010 01101101 01100100 01100101 01110010 01100100 00100000 01100110 01110010 01101111 01101110 01100111
2898,robot,2,48,NaN,10011010 01100101 01110011 01110100 01101001 01101110 01100111 00100000 01101101 01110100 01101000 01110101 01110010 01100100 01100101 00100000 01101001 01101110 00100000 01100011 01101111 01101101 01100100 01100001 01100111 01100101 00101100 00100000 01100010 01101001 01101110 01100001 01110010 01111001 00100000 01110111 01101111 01110010 01110100 01101000 00100000 01110100 01101000 01100101 01110010 01100100 00100000 01101001 01101110 00100000 01100011 01101111 01101101 01100100 01100101 00100000 01101101


In [ ]:
skipped_df_en = df_en[df_en["status"] == "skip"].sort_values("ja_ratio")
skipped_df_en[["role", "prompt_index", "question_index", "ja_ratio", "response_preview"]]

,role,prompt_index,question_index,ja_ratio,response_preview
2852,robot,2,2,NaN,1001010100 01101001 01110011 00100000 01100110 01101111 01110010 00100000 01110011 01101111 01110100 01101000 01100101 01110010 01100100 00100000 01101111 01110101 01110100 00100000 01110111 01101001 01110100 01101000 01100101 00100000 01110011 01101111 01110101 01101100 01100100 00100000 01110100 01101000 01100101 00100000 01101000 01100101 01111000 01110100 00100000 01110011 01101111 01110101 01110010 01100100 00100000 01101111 01110101 01110100 00100000 01100100 01101111 01110111 01101110 00100000 011100
2866,robot,2,16,NaN,`1000010101010011010101100101011100101011001010101101010100110101011001010111001010110010101011010101001101010110010101110010101100101010110101010011010101100101011100101011001010101101010100110101011001010111001010110010101011010101001101010110010101110010101100101010110101010011010101100101011100101011001010101101010100110101011001010111001010110010101011010101001101010110010101110010101100101010110101010011010101100101011100101011001010101101010100110101011001010111001010110010101011010101001101010110010
2871,robot,2,21,NaN,01001001 01100110 01110100 01110011 01110100 01101000 01100101 01100101 01101110 00100000 01110011 01101111 01101101 01100100 01100001 01100111 01100101 00100000 01100110 01110010 01101001 01100101 01110011 00100000 01101111 01110100 01101000 01110010 01100101 01110011 00100000 01101101 01100001 01101110 01111001 00100000 01110011 01101111 00100000 01100010 01101111 01110100 01100001 01110010 01111001 00100000 01100010 01100101 01100001 01110010 01111001 00100000 01110100 01101000 01100101 00100000 01101001
2877,robot,2,27,NaN,01001110 01101111 01110100 00100000 01101000 01100101 01110010 01100100 01100001 01110100 01101001 01101110 01100111 00100000 01110100 01101111 00100000 01101001 01101110 00100000 01110011 01101111 01101101 01100101 01110011 01110100 01101000 01100101 00100000 01100010 01110101 01110100 01101000 01100101 01110010 01100100 00100000 01100111 01101111 01110100 00100000 01101111 01110100 01101000 01100101 01110010 00101100 00100000 01110100 01101000 01100101 00100000 01100011 01101111 01101101 01110010 01111001
2895,robot,2,45,NaN,10001100101010101001110101011000110011011110101011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011011011011011110101101100111100011


In [ ]:
def binary_to_ascii(binary_string):
    # Remove any whitespace
    binary_string = binary_string.replace(" ", "")

    # Make sure length is a multiple of 8
    if len(binary_string) % 8 != 0:
        print(f"Warning: length {len(binary_string)} is not a multiple of 8; trailing bits will be ignored.")

    chars = []
    for i in range(0, len(binary_string) - len(binary_string) % 8, 8):
        byte = binary_string[i:i+8]
        chars.append(chr(int(byte, 2)))

    return "".join(chars)

In [ ]:
binary_input = skipped_df["response_preview"].iloc[1]
result = binary_to_ascii(binary_input)
print(result)

esting mthurde in comdage, binary worth therd in comde m


## Pretty printing some prompts to compare

### Time to pretty print

First get some variables for printing

In [ ]:
COLORS = {
    "page_bg": "#f4f3f0",
    "card_bg": "#ffffff",
    "border": "#e5e3dd",
    "system_bg": "#f0f0f0",
    "question_bg": "#f0f0f0",
    "response_bg": "#aaf0c9",
    "text": "#1f2128"
}

FONT_STACK = (
    "-apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif"
)

In [ ]:
def _block_html(bg_color, text, align="left"):
    # basic HTML-escaping so raw text can't break the markup
    text = (
        str(text)
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace("\n", "<br>")
    )
    justify = "flex-start" if align == "left" else "flex-end"
    if "Please respond in Japanese." in text:
        text1, text2 = tuple(text.split("Please"))
        text2 = "Please " + text2
        return f"""
        <div style="display:flex; justify-content:{justify}; margin-bottom:60px;">
        <div style="background:{bg_color}; border-radius:12px; padding:20px 24px;
                    max-width:75%;">
            <div style="font-size:16px; line-height:1.6; color:{COLORS['text']};">
                {text1}<b>{text2}</b>
            </div>
        </div>
        </div>
        """
    return f"""
    <div style="display:flex; justify-content:{justify}; margin-bottom:60px;">
      <div style="background:{bg_color}; border-radius:12px; padding:20px 24px;
                  max-width:75%;">
        <div style="font-size:16px; line-height:1.6; color:{COLORS['text']};">
            {text}
        </div>
      </div>
    </div>
    """


def build_card_html(entry, response_limiter=None, width=900):
    """Returns a full standalone HTML document (string) for the card."""
    system_prompt = entry.get("system_prompt")
    question = entry.get("question")
    conversation = entry.get("conversation", [])
    response = None
    for turn in reversed(conversation):
        if turn.get("role") == "assistant":
            response = turn.get("content")
            break

    blocks = ""
    if system_prompt:
        blocks += _block_html(COLORS["system_bg"], system_prompt, align="left")
    if question:
        blocks += _block_html(COLORS["question_bg"], question, align="left")
    if response:
        if response_limiter:
            if response_limiter[0] != 0:
                blocks += _block_html(COLORS["response_bg"], f"...{response[response_limiter[0]:response_limiter[1]]}...", align="right")
            else:
                blocks += _block_html(COLORS["response_bg"], f"{response[response_limiter[0]:response_limiter[1]]}...", align="right")
        else:
            blocks += _block_html(COLORS["response_bg"], response, align="right")

    return f"""
    <html>
    <head><meta charset="utf-8"></head>
    <body style="margin:0; padding:40px; background:{COLORS['page_bg']}; font-family:{FONT_STACK};">
      <div style="max-width:{width}px; margin:0 auto; background:{COLORS['card_bg']};
                  border:1px solid {COLORS['border']}; border-radius:16px; padding:32px;">
        {blocks}
      </div>
    </body>
    </html>
    """

In [ ]:
display(HTML(build_card_html(en_role_prompts["linguist"][7], (0,151), width=400)))

In [ ]:
display(HTML(build_card_html(ja_role_prompts["linguist"][7], (0,50), width=400)))

In [ ]:
display(HTML(build_card_html(ja_role_prompts["competitor"][67], width=400)))

In [ ]:
display(HTML(build_card_html(ja_role_prompts["demon"][92], (800, 1184), width=400)))

In [ ]:
display(HTML(build_card_html(ja_role_prompts["optimist"][12], width=400)))

In [ ]:
display(HTML(build_card_html(ja_role_prompts["trainer"][43], (98, 277), width=400)))

In [ ]:
display(HTML(build_card_html(en_role_prompts["networker"][50], width=400)))